# Day 2 lab pipeline runbook

This notebook runs the lab pipeline in the right order. It runs **only what needs to run**, checks each step before moving on, and **records every run** in Snowflake.

## Where things live
| What | Where |
| --- | --- |
| Shared values: role, database, warehouse, schema, program, slugs, models, cutoffs | `sql/runbook_config.sql`. Change values there and nowhere else. This notebook runs it first. |
| The SQL itself | `sql/01_setup.sql` … `sql/13_video_search.sql`. The notebook runs **sections** of these files between `-- @runbook:begin` / `-- @runbook:end` markers and never keeps its own copy of the SQL. |
| The engine (parsing, planning, fingerprints, recording) | `lab_metadata/runbook_lib.py` |
| Run history | `runbook_runs`, `runbook_step_runs`, `runbook_source_freshness`, `retrieval_scores_history`, `slide_descriptions_history` |

## Section roles
| Role | What the notebook does |
| --- | --- |
| **BUILD** / **DEFINE** | Runs in the step's **build cell**, only when the run plan says RUN |
| **CHECK** | Runs right after the build and stops the run if an expectation fails (for example *expect zero rows*). Results appear in the **check cell** |
| **OUTPUT** | Key results, shown in the **output cell** |
| **EXPLORE** | Never run here. Listed in the Explore index with file and line numbers, so you can run them from the file when they're useful |
| **EXPERIMENT** | Your hand-run before/after statements. The notebook runs label-based versions of the same logic in the Experiment section |

## Pipeline order (by dependency, not file number)
```
01 setup -> 02/08/10/12 procedures
03a catalog -> 03d lessons (external API)
   04b chunk transcripts            05a parse slides (AI) -> 05c slide pages
                                    11a render pages -> 11b image embeddings (AI) -> 11e descriptions (AI)
   06a corpus (transcripts + slides + descriptions + FAQs)  <- needs 11e
   06b text embeddings (AI) -> 07a Cortex Search service
   09b eval set -> 09c VECTOR + CORTEX_SEARCH -> 11d IMAGE -> 09d scores
13 video path (optional: INCLUDE_VIDEO)
```
Two pipelines read the same slides. The **text pipeline** (05 → 06 → 07/09) only sees words that `AI_PARSE_DOCUMENT` pulled out. The **image pipeline** (11a/11b → IMAGE) sees the picture. `AI_COMPLETE` in 11e connects them: it writes a description of each picture, and 06a appends it to the slide's text.

## How the run plan decides
For every step it compares four things with what the step used the last time it succeeded:
1. **its SQL**, hashed with comments and whitespace ignored
2. **its inputs**, as a row count plus a content hash of each table, view, stage or service it reads
3. **its settings** from `runbook_config.sql`
4. **its outputs**: if one is missing the step runs; if one changed outside the runbook it's flagged as DRIFT

With no history yet (the first run), each step runs a **consistency check** that compares its output with its inputs directly. When a step runs, everything downstream of it is re-checked. If an upstream step produced identical content, the downstream steps skip.

Cost and external steps are **never blocked** because of cost. They run when the plan shows something changed. Set `REQUIRE_APPROVAL_FOR_COST=True` to stop and review first.

## Next-time checklist (new experiment)
1. Set `MODE='full_experiment'`, a new `LABEL` (for example `PROMPT_V2`), and a `CHANGE_NOTE`.
2. Run down to **Experiment: snapshot the baseline** *before* changing anything. It copies the corpus, embeddings, question embeddings and scores to `*_<label>`.
3. Make your change (SQL file or config value), rerun from **Run plan** onward, and read why each step runs.
4. The Experiment section builds a baseline search service on the snapshot, scores both versions on the same questions, and compares ranks, scores and margins.
5. Set `CONFIRM_DROP=True` and run the last cell to drop the baseline service (it refreshes daily until you drop it).

# Section 1: Control panel and guardrails
Run options, shared config (`sql/runbook_config.sql`), recording tables, source freshness and the run plan. The plan sets `rb.halted_reason` when the run must stop (DRY_RUN, cost approval, `$eval_top_k` below 10); every later cell checks it.

## Options reference
The next cell sets **how this run behaves**. Shared **values** (models, slugs, program, cutoffs, lesson cap, golden version, judge) are not set here: they live in `sql/runbook_config.sql`. The engine checks `MODE`, `REFRESH_SOURCES` and `FORCE_STEPS` in `Runbook.make_plan()` (`lab_metadata/runbook_lib.py`) and stops with an error on an invalid value.

### Run mode and experiment
| Option | Allowed values | What it does |
| --- | --- | --- |
| `MODE` | `"maintenance"` | Builds whatever changed. No baseline, no before/after scoring. Scores go to `retrieval_question_scores_history` with an empty label. |
| | `"full_experiment"` | Before building, copies `search_corpus`, `text_embeddings`, `eval_query_embeddings`, `retrieval_results` and the per-question ranks to `*_<label>` (cell **Snapshot the baseline**). Then it builds your change, scores the baseline on the same questions, and compares. |
| | `"score_existing"` | The change is already built and the `*_<label>` baseline already exists. Only scores and compares. Fails if `search_corpus_<label>` or `text_embeddings_<label>` is missing. |
| `LABEL` | Upper-case letters, digits, `_` (max 41), e.g. `"BEFORE_DESC"`, `"PROMPT_V2"` | Names the baseline: tables `*_<label>`, the service `course_search_<label>`, and the methods `VECTOR_<LABEL>` / `CORTEX_SEARCH_<LABEL>`. Ignored in `maintenance`. |
| `CHANGE_NOTE` | Any text | What this run tests or why it runs. Saved in `runbook_runs.change_note`. |
| `EXPERIMENT_NAME` | Any text, or `""` | A short label for the run (e.g. `"1_chunk_headers"`), saved in `runbook_runs` and the history fact and used as the X axis of the progression chart. Leave it `""` and it becomes the run date (e.g. `"2026-11-25"`). It is only a label: what the run is compared with comes from the champion, never from this name. |

**Champion (not an option).** Every run is compared question by question (win/loss/tie + sign test) with the **champion**: the most recently promoted run in `runbook_runs` (`promoted_at`, `promoted_reason`). The run records it in `runbook_runs.baseline_run_id`. The champion changes only in the cell **Champion: promote a run**, after a verdict says a run beat it. If the champion's eval set fingerprint differs from this run's, the comparison is skipped and recorded as *not comparable*.

### Control
| Option | Allowed values | What it does |
| --- | --- | --- |
| `DRY_RUN` | `True` / `False` | `True`: show the freshness panel and the run plan, then stop. `make_plan()` sets `rb.halted_reason`, every later cell prints *Skipped*, and the engine refuses build and scoring calls. The run is recorded with status `DRY_RUN`. |
| `FORCE_STEPS` | List of step IDs from: `01 02 08 10 12 03a 03d 04b 05a 05c 11a 11b 11e 06a 06b 07a 09b 09c 11d 09d 09e 13a 13b 13c 13d 13f` | Runs those steps whatever the plan says. Steps downstream are rechecked and run if their inputs changed. An unknown ID (e.g. `"3d"` instead of `"03d"`) stops the run. Video steps are ignored while `INCLUDE_VIDEO=False`, and 09e while `INCLUDE_ANSWER_EVAL=False`. |
| `REQUIRE_APPROVAL_FOR_COST` | `True` / `False` | `True`: if any step that will run costs AI credits or calls the API, stop after the run plan (status `AWAITING_APPROVAL`). Review the plan, set `False`, rerun from Setup. |
| `CONTINUE_ON_CHECK_FAIL` | `True` / `False` | `False`: a failed check (e.g. *expect zero rows* returned rows) stops the run before the next step. `True`: records the failure in `runbook_step_runs.checks` and continues. Use `True` only for debugging. |
| `INCLUDE_VIDEO` | `True` / `False` | Turns on steps 12 and 13. **Cost:** 13b downloads every program video (hundreds of MB each); 13c runs `AI_MULTI_EMBED`, billed by video length; 13f embeds the video questions. |
| `INCLUDE_ANSWER_EVAL` | `True` / `False` | Turns on step 09e: one `AI_COMPLETE` answer (`$answer_model`) and one judge call (`$judge_model`) per eval question. Run once with `DRY_RUN=True` to see the estimate. |

The engine also halts (status `HALTED_CONFIG`) if `$eval_top_k` in `sql/runbook_config.sql` is below 10, because every metric is computed at @10.

### Source freshness (external ingest: steps 03a, 03d, 13b)
| Option | Allowed values | What it does |
| --- | --- | --- |
| `PROBE_CATALOG` | `True` / `False` | `True`: the freshness cell calls `ingest_catalog()` (3 API requests) and compares the lesson list before and after. `False`: no API calls; only the coverage and age signals count. |
| `REFRESH_SOURCES` | `"auto"` | Ingest (03d, plus 13b when video is on) only if the freshness cell's recommendation is **REFRESH**. |
| | `True` | Always ingest this run. |
| | `False` | Never ingest this run (e.g. during an experiment, so new lessons don't mix into the comparison). |
| `SOURCE_MAX_AGE_DAYS_ACTIVE` | Whole days | While the program runs (start date to end date + `ACTIVE_GRACE_DAYS`), and before it starts, recommend REFRESH once the last ingest is older than this. |
| `SOURCE_MAX_AGE_DAYS_INACTIVE` | Whole days or `None` | Same rule after the program ends. `None` = no age rule; only catalog changes and new gaps count. |
| `ACTIVE_GRACE_DAYS` | Whole days | Days after the end date that still count as active (recordings often arrive late). |

### Cleanup
| Option | Allowed values | What it does |
| --- | --- | --- |
| `CONFIRM_DROP` | `True` / `False` | `True`: the last cell drops `course_search_<label>`. It bills serving credits while it exists, even with no queries. |

### Which mode?
| You changed | Mode |
| --- | --- |
| Nothing yet; you want new lessons, rebuilds after a source refresh, or a routine check | `maintenance` |
| Something that can change **retrieval quality**, and you want to measure it: corpus text (step 4 chunk sizes, 5 parse options, 6a `search_text`, 11e prompt or `$describe_max_chars`), `$text_embed_model`, the 07a service definition | `full_experiment` with a new `LABEL`, set **before** you run the change |
| Only the eval set (`eval_questions` / `eval_expected` in 9b, or `$golden_version`) | `maintenance`. The ruler changed, not the system. Comparing old and new scores would mix two question sets (the eval fingerprint in the report shows this). |
| `$eval_top_k` | `maintenance`. Scores at a different k aren't comparable. |
| Lesson ingest (new transcripts or decks) | `maintenance`. Content was added, not improved, and the before and after corpora hold different material. |
| `$answer_model`, `$gen_top_k`, or the judge settings | `maintenance` with `INCLUDE_ANSWER_EVAL=True`. The answers are compared with the champion's. |

**One experiment = one change**, measured against a baseline taken just before it. A good `LABEL` names the state the baseline captures (e.g. `BEFORE_DESC`, `BEFORE_CHUNK800`, `BEFORE_PROMPT_V2`). Use a new label each time. If the label already exists, the snapshot cell keeps the old copy and says *already exists, kept*, so you would compare against an older state.

**Waterfall order.** Lock in the winner of one stage before starting the next: 1. chunk headers and chunk size (characters), 2. embedding model, 3. `$gen_top_k` (after 09e). VECTOR vs CORTEX_SEARCH is compared on every run already.

In [ ]:
# ---- Options for this run: allowed values and effects are in the "Options reference" cell above ----
# Shared VALUES (models, slugs, program, lesson cap) live in sql/runbook_config.sql, not here.
OPTIONS = {
    # score_existing : the change is already live and the baseline (*_<LABEL>) exists; score and compare
    # full_experiment: snapshot the baseline first, then apply the change and compare
    # maintenance    : run whatever changed, no experiment sections
    "MODE": "maintenance",                  #"full_experiment",  #"score_existing",
    "LABEL": "BEFORE_DESC_V2",              # baseline suffix: search_corpus_before_desc_v2, CORTEX_SEARCH_BEFORE_DESC_V2, ... (ignored in maintenance)
    #"LABEL": "BEFORE_DESC",
    "CHANGE_NOTE": "Verification run: nothing changed since champion 9dfccaf7f3c4 (desc v2 image-aware); expect every step to skip and all ties vs the champion",
    #"CHANGE_NOTE": "Clear 09d DRIFT left by the desc v2 experiment (baseline rows were added after 09d recorded its outputs); rebuild 09d views only",
    #"CHANGE_NOTE": "11e describe rule v2: describe a slide if it has an image tag OR its real text (image links and footer stripped) is under $describe_max_chars; 13 -> 63 slides described",
    #"CHANGE_NOTE": "09b simplified: eval_questions loaded directly (golden_seed_v1 / eval_questions_golden removed); same 21 questions",
    #"CHANGE_NOTE": "Verification run: nothing changed since champion 287e36936493; expect every step to skip and all ties vs the champion",
    #"CHANGE_NOTE": "Phase baseline: golden set v1, @10 metrics, scores, question-grain history, sign test, token cost, 09e judge, video",
    #"CHANGE_NOTE": "slide descriptions (11e AI_COMPLETE) appended to slide search_text in 06a; Q17-Q21 added",
    "EXPERIMENT_NAME": "",                  # free label, e.g. "1_chunk_headers"; "" = the run date (e.g. "2026-11-25")
    #"EXPERIMENT_NAME": "desc_v2_image_aware",
    # The baseline to compare against is the champion in runbook_runs (cell 'Champion: promote a run'), not an option.

    "DRY_RUN": False,                       # True = show the run plan, run nothing
    "FORCE_STEPS": [],                      # e.g. ["07a"] to rebuild a step regardless of the plan;
                                            # (step IDs are two-digit: use "03d"; an unknown ID stops the run)
    "REQUIRE_APPROVAL_FOR_COST": False,     # True = stop before AI / external steps so you can review the plan
    "CONTINUE_ON_CHECK_FAIL": False,        # False = a failed check stops the run before the next step
    "INCLUDE_VIDEO": True,                 # step 13 video path (costs: video downloads + AI_MULTI_EMBED by video length)
    "INCLUDE_ANSWER_EVAL": True,            # step 09e: AI_COMPLETE answer + judge per eval question (2 calls each)

    # Source freshness (see the freshness cell)
    "PROBE_CATALOG": True,                  # 3 small API requests to see if the lesson catalog changed
    "REFRESH_SOURCES": "auto",               # False,   'auto' = ingest only if the freshness panel says REFRESH
    "SOURCE_MAX_AGE_DAYS_ACTIVE": 7,        # while the program runs (+ ACTIVE_GRACE_DAYS): refresh if older
    "SOURCE_MAX_AGE_DAYS_INACTIVE": None,   # after the program: no age-based refresh
    "ACTIVE_GRACE_DAYS": 7,

    "CONFIRM_DROP": False,                  # True = the cleanup cell drops the baseline search service
}

In [ ]:
import importlib, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
from snowflake.snowpark.context import get_active_session

# Find the workspace root (the folder that holds sql/runbook_config.sql)
WORKSPACE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sql" / "runbook_config.sql").exists())
sys.path.insert(0, str(WORKSPACE / "lab_metadata"))
import runbook_lib
importlib.reload(runbook_lib)
pd.set_option("display.max_colwidth", 200)

session = get_active_session()
rb = runbook_lib.Runbook(session, WORKSPACE, OPTIONS)
config_table = rb.apply_config()          # runs sql/runbook_config.sql: the single source of shared values
rb.ensure_record_tables()                 # also moves the old method-grain history to retrieval_scores_history_legacy once
rb.start_run()
RUN_ID = rb.run_id                        # used by SQL cells as {{RUN_ID}}

display(Markdown(f"**Run `{rb.run_id}`** · mode `{OPTIONS['MODE']}` · label `{OPTIONS['LABEL']}` · "
                 f"{len(rb.sections)} sections parsed from `{WORKSPACE.name}/sql`"))
display(config_table)
lint = rb.lint()
if lint:
    display(Markdown("**Config mismatches in SQL that can't read session variables:**\n" + "\n".join(f"- {i}" for i in lint)))
else:
    display(Markdown("Config check: the 07a service DDL matches `runbook_config.sql`."))

## Explore index
The runbook doesn't run these queries. They're educational and exploratory: sample searches, previews, single-row tests and the `ask_course` demos. Open the file at the line range shown and run them when they help. Some cost AI credits (the `cost` column). The second table lists your hand-run experiment statements, which the Experiment section replaces with label-based versions.

In [ ]:
display(Markdown("**EXPLORE sections (spot-run from the file)**"))
display(rb.catalog(roles={"EXPLORE"}))
display(Markdown("**EXPERIMENT sections (hand-run versions; the Experiment section below runs label-based equivalents)**"))
display(rb.catalog(roles={"EXPERIMENT"}))

## Source freshness panel: should the external ingest run?
The **freshness panel** is the output of the next cell. That cell calls `Runbook.check_freshness()` in `lab_metadata/runbook_lib.py`, which prints a **recommendation** (`REFRESH` or `NOT NEEDED`) with the reason, then a table of signals. With `REFRESH_SOURCES="auto"`, the run plan uses that recommendation to decide steps 03d and 13b (`Runbook.make_plan()`).

**Signals, cheapest first:**
1. **Catalog probe** (`PROBE_CATALOG=True`). Keeps the current lesson list for `$program_id` in memory (`before`), calls `ingest_catalog()` (3 API requests, which write a new row to `api_responses`), reads the list again (`after`), and compares. Rows `new_lessons`, `removed_lessons`, `changed_lessons` (title, slide flag or recording length). The comparison is against the previous fetch, so a change shows up on the first run after it happens. The next run's `before` already includes it.
2. **Coverage gaps.** Lessons with a recording length but no transcript loaded (`recorded_but_no_transcript`), lessons flagged with slides but no deck (`slides_but_no_deck`), and lessons past `$ingest_max_lessons` (`beyond_ingest_cap`). A gap triggers REFRESH only until a successful 03d ingest has tried it. After that it's listed under `gaps_already_tried_by_last_ingest` and no longer counts: the API had nothing for it then.
3. **Age, tied to the program calendar.** `ingest_age_days` against `SOURCE_MAX_AGE_DAYS_ACTIVE` / `_INACTIVE` for the current `phase`.

**Where to review it later:** every check, including DRY_RUN, is saved with the run:
```sql
SELECT run_id, checked_at, phase, recommendation, reason, details
FROM runbook_source_freshness ORDER BY checked_at DESC;
```
A refresh is safe: unchanged decks are skipped by sha256, and rewritten transcripts only make the AI steps rerun if their content actually changed.

In [ ]:
recommendation, reason, freshness_table = rb.check_freshness()
display(Markdown(f"### {recommendation}\n{reason}"))
display(freshness_table)

## Run plan
One row per step: **RUN**, **SKIP**, **DRIFT** (an output changed outside the runbook; add the step to `FORCE_STEPS` to rebuild it) or **NOT_INCLUDED**, with the reason, cost type and an estimated call count. Every decision, including skips, is recorded in `runbook_step_runs`.

In [ ]:
saved = rb.ensure_descriptions_saved()   # keep a copy of AI_COMPLETE descriptions the runbook hasn't seen yet
if saved:
    display(Markdown(f"Saved {saved} current slide descriptions to `slide_descriptions_history` before anything can replace them."))
plan_table = rb.make_plan()               # also sets rb.halted_reason (DRY_RUN, cost approval, $eval_top_k < 10)
display(plan_table)
runs = plan_table[plan_table.decision == "RUN"]
costly = runs[runs.cost != "none"]
display(Markdown(f"**{len(runs)} steps will run**, {len(costly)} of them with AI or external cost: "
                 + (", ".join(f"{r.step} ({r.estimate or r.cost})" for r in costly.itertuples()) or "none")))
# HALTED stops every later build, snapshot and experiment cell, even under Run All.
# The engine owns the flag (rb.halted_reason) and refuses build/scoring calls while it is set.
HALTED = rb.halted_reason
if HALTED:
    display(Markdown(f"### Stopped\n{HALTED}"))

In [ ]:
CHECK_RESULTS, CHECK_FRAMES = {}, {}

def halted():
    """True when the run plan halted this run (DRY_RUN, REQUIRE_APPROVAL_FOR_COST, or a config guard)."""
    reason = rb.halted_reason if "rb" in globals() else "Setup has not run yet"
    if reason:
        display(Markdown(f"Skipped: {reason}"))
    return bool(reason)

def build_stage(steps):
    """Build cell: run each step the plan says RUN, then its checks right away as a gate."""
    if halted():
        return
    for step in steps:
        if step not in rb.plan:
            continue
        rb.execute_step(step, display=lambda m: display(Markdown(m)))
        frames = []
        CHECK_RESULTS[step] = rb.check_step(step, display_frame=lambda s, c, f: frames.append((s, c, f)))
        CHECK_FRAMES[step] = frames

def show_checks(steps):
    """Check cell: pass/fail per check, plus the rows each check returned (first 20)."""
    for step in steps:
        if step not in CHECK_RESULTS or CHECK_RESULTS[step].empty:
            continue
        display(Markdown(f"**{step} checks**"))
        display(CHECK_RESULTS[step])
        for section, check, frame in CHECK_FRAMES.get(step, []):
            if frame is not None and len(frame):
                display(Markdown(f"`{section.id}` statement {check['statement']} ({check['detail']}) - {section.location}"))
                display(frame.head(20))

def show_outputs(steps):
    """Output cell: the OUTPUT sections of each step."""
    if rb.halted_reason:
        return
    for step in steps:
        if step not in rb.plan or rb.plan[step].action == "NOT_INCLUDED":
            continue
        for section, frame in rb.output_step(step):
            display(Markdown(f"**{section.id}** - {section.purpose} ({section.location})"))
            display(frame if frame is not None else "(no rows)")

## Experiment: snapshot the baseline (`full_experiment` only)
Run this **before** you change anything. It clones `search_corpus`, `text_embeddings`, `eval_query_embeddings` and `retrieval_results`, and saves `retrieval_question_scores`, each to `*_<label>`. An existing snapshot is never overwritten, so rerunning is safe. In `score_existing` mode the baseline already exists (for `BEFORE_DESC`: `search_corpus_before_desc`, `text_embeddings_before_desc`, `course_search_before_desc`), and this cell only reports it.

In [ ]:
if halted():
    pass
elif OPTIONS["MODE"] == "full_experiment":
    for line in rb.snapshot_baseline():
        display(Markdown(f"- {line}"))
elif OPTIONS["MODE"] == "score_existing":
    names = rb.baseline_names()
    rb.refresh_kinds()
    found = {k: v in rb._object_kinds() for k, v in names.items()}
    display(pd.DataFrame([(k, names[k], found[k]) for k in names], columns=["part", "object", "exists"]))
    if not (found["corpus"] and found["embeddings"]):
        raise RuntimeError("score_existing needs the baseline corpus and embeddings for this LABEL")
else:
    display(Markdown("maintenance mode: no experiment"))

# Section 2: Data prep (ingest, chunk, parse, embed)
Stages A to E build everything retrieval reads: procedures, lesson ingest, transcript chunks, slide text and descriptions, the search corpus, its embeddings and the Cortex Search service. The data check after Stage E stops a bad change (e.g. slide rows dropping to 0) before it reaches the scores.

## Stage A: setup objects and procedures (steps 01, 02, 08, 10, 12)
`01` creates stages, tables (`IF NOT EXISTS`) and catalog views. `02`, `08`, `10` and `12` create the procedures. Procedure steps run when their source in the file changed, or when the signature in Snowflake doesn't match the file. For example, `run_search_eval` now takes `service_name` and `method_label`.

In [ ]:
STAGE_A = ["01", "02", "08", "10", "12"]
build_stage(STAGE_A)

In [ ]:
show_checks(STAGE_A)

## Stage B: external ingest (steps 03a, 03d)
Runs only when the freshness panel recommends a refresh (or `REFRESH_SOURCES=True`). `03a` already ran as the catalog probe if `PROBE_CATALOG=True`.

In [ ]:
STAGE_B = ["03a", "03d"]
build_stage(STAGE_B)

In [ ]:
show_checks(STAGE_B)

In [ ]:
show_outputs(STAGE_B)

## Stage C: transcript chunks and slide text (steps 04b, 05a, 05c)
`04b` splits transcripts into overlapping chunks. `05a` runs `AI_PARSE_DOCUMENT` only on decks not parsed yet. `05c` turns parses into one row per slide page.

In [ ]:
STAGE_C = ["04b", "05a", "05c"]
build_stage(STAGE_C)

In [ ]:
show_checks(STAGE_C)

In [ ]:
show_outputs(STAGE_C)

## Stage D: slide images and descriptions (steps 11a, 11b, 11e)
`11a` renders each page to an image. `11b` embeds the images (IMAGE method). `11e` has `AI_COMPLETE` describe text-light slides, which 06a then appends to the slide text. 11e writes **different text every run**, so it runs only when its inputs, SQL or settings (`$describe_model`, `$describe_max_chars`) change. Each run's descriptions are saved to `slide_descriptions_history`.

In [ ]:
STAGE_D = ["11a", "11b", "11e"]
build_stage(STAGE_D)

In [ ]:
show_checks(STAGE_D)

In [ ]:
show_outputs(STAGE_D)

## Stage E: corpus, text embeddings, search service (steps 06a, 06b, 07a)
`06a` combines transcript chunks, slide pages (plus the 11e descriptions) and FAQs into `search_corpus`. `06b` embeds every row (VECTOR method). `07a` rebuilds the `course_search` service (CORTEX_SEARCH method).

In [ ]:
STAGE_E = ["06a", "06b", "07a"]
build_stage(STAGE_E)

In [ ]:
show_checks(STAGE_E)

In [ ]:
%%sql -r corpus_by_source_type
-- Section 2 gate: row counts and text length per source type in the corpus that retrieval reads.
-- A change that drops a source type to 0 rows, or makes chunks balloon, shows up here before scoring.
SELECT c.source_type,
       COUNT(*) AS corpus_rows,
       COUNT(DISTINCT c.lesson_slug) AS lessons,
       ROUND(AVG(LENGTH(c.search_text))) AS avg_chars,
       MIN(LENGTH(c.search_text)) AS min_chars,
       MAX(LENGTH(c.search_text)) AS max_chars,
       COUNT_IF(e.embedding IS NULL) AS rows_without_embedding
FROM search_corpus c
LEFT JOIN text_embeddings e ON e.source_id = c.source_id
GROUP BY c.source_type
ORDER BY c.source_type;

# Section 3: Retrieval evaluation (@1, @5, @10)
Stage F scores the golden questions with VECTOR, CORTEX_SEARCH and IMAGE. Every metric is cut at rank 10. The sanity check after it shows the questions no method found, with what came back first instead. Stage G (video) and the Experiment cells follow.

## Stage F: evaluation (steps 09b, 09c, 11d, 09d)
`09b` loads the eval questions into `eval_questions` (with their `question_type` / `source_type` labels; only versions up to `$golden_version`), plus the answer list. Editing a question changes the eval fingerprint, so later comparisons with runs on the old questions say "not comparable". `09c` ranks the top k with VECTOR and CORTEX_SEARCH and stores each result's score. `11d` ranks slide questions with IMAGE. `09d` rebuilds the score views: every metric is cut at rank 10 (hit@1, hit@5, MRR@10, recall@10, nDCG@10), and `retrieval_scores_by_slice` shows each `question_type` / `source_type` slice with its question count. Any method label in `retrieval_results`, including the baseline labels from the Experiment section, shows up in the scores automatically.

In [ ]:
STAGE_F = ["09b", "09c", "11d", "09d"]
build_stage(STAGE_F)

In [ ]:
show_checks(STAGE_F)

In [ ]:
show_outputs(STAGE_F)

In [ ]:
# Questions a method did not find in the top 10 (or found lowest), with the #1 result it returned instead.
# Three of these usually tell you whether search is broken or the question wording is ambiguous.
if not halted():
    worst = session.sql("""
        SELECT s.method, s.question_id, s.question, s.question_type, s.first_relevant_rank,
               c.source_type AS top1_type, c.citation AS top1_citation, LEFT(c.search_text, 160) AS top1_text
        FROM retrieval_question_scores s
        LEFT JOIN retrieval_results r ON r.method = s.method AND r.question_id = s.question_id AND r.result_rank = 1
        LEFT JOIN search_corpus c ON c.source_id = r.source_id
        WHERE s.method IN ('VECTOR', 'CORTEX_SEARCH')
        QUALIFY ROW_NUMBER() OVER (PARTITION BY s.method ORDER BY COALESCE(s.first_relevant_rank, 99) DESC, s.question_id) <= 3
        ORDER BY s.method, COALESCE(s.first_relevant_rank, 99) DESC""").to_pandas()
    display(Markdown("**3 worst questions per text method** (`FIRST_RELEVANT_RANK` empty = not in the top 10)"))
    display(worst)

## Stage G: video (optional, steps 13a-13f)
Runs only with `INCLUDE_VIDEO=True`. `13b` downloads every program video again to hash it, and `13c` is billed by video length.

In [ ]:
STAGE_G = ["13a", "13b", "13c", "13d", "13f"]
build_stage(STAGE_G)
show_checks(STAGE_G)
show_outputs(STAGE_G)

## Experiment: score the baseline
Builds (or reuses) a Cortex Search service on the baseline corpus, `course_search_<label>`, and scores the **same current questions** against the baseline:
- `CORTEX_SEARCH_<LABEL>` uses `run_search_eval($eval_top_k, 'course_search_<label>', 'CORTEX_SEARCH_<LABEL>')`
- `VECTOR_<LABEL>` ranks the current question embeddings against `text_embeddings_<label>`

The only difference between before and after is then the corpus text, because the questions, models and top k are the same. IMAGE doesn't read the corpus, so it serves as the control.

In [ ]:
if halted():
    pass
elif OPTIONS["MODE"] in ("score_existing", "full_experiment"):
    display(Markdown(rb.ensure_baseline_service()))
    display(pd.DataFrame(rb.score_baseline()))
    rb.run_section(rb.sections["09d.build"])          # views pick up the new method labels
    rb.run_section(rb.sections["09d.build_scores"])
else:
    display(Markdown("maintenance mode: no experiment"))

## Experiment: compare
1. **Ranks per question.** `NULL` means no correct answer in the top k. Lower is better. `image_control` should not move.
2. **Scores per method.** hit@1, hit@5 and MRR before and after.
3. **What changed per answer, with similarity and margin (VECTOR).** `change_applied` is derived from the data: it compares the stored before and after `search_text` and reports exactly what was added. `margin` is the correct answer's similarity minus the best wrong item's similarity. Above 0 means it ranked first, and a bigger value means a safer win. Unchanged answers are controls: their `sim_change` should be 0, although their margin can still move when a competing item changed.

In [ ]:
if halted():
    pass
elif OPTIONS["MODE"] in ("score_existing", "full_experiment"):
    ranks = rb.compare_ranks()
    display(Markdown("**1. First relevant rank per question, before vs after**"))
    display(ranks)
    moved = ranks[(ranks.VECTOR_BEFORE.fillna(99) != ranks.VECTOR_AFTER.fillna(99)) |
                  (ranks.SEARCH_BEFORE.fillna(99) != ranks.SEARCH_AFTER.fillna(99))]
    display(Markdown(f"{len(moved)} questions changed rank in VECTOR or CORTEX_SEARCH: " + ", ".join(moved.QUESTION_ID)))
    display(Markdown("**2. Scores per method**"))
    display(rb.compare_scores())
    display(Markdown("**3. What changed per answer, similarity and margin (VECTOR)**"))
    display(rb.compare_margins())

# Section 4: Generation and LLM-as-judge (optional)
Stage H runs step `09e` only with `INCLUDE_ANSWER_EVAL=True`. Every eval question is answered the way `ask_course` answers (Cortex Search top `$gen_top_k`, then `AI_COMPLETE` with `$answer_model`), and a pinned judge (`$judge_model`, temperature `$judge_temperature`, prompt `$judge_prompt_version`) grades groundedness, answer relevance and context relevance from 1 to 5, and whether the answer refused. Q16 should be refused. The cost cell after it shows measured tokens and elapsed time for this run.

In [ ]:
STAGE_H = ["09e"]
if "build_stage" in globals() and "rb" in globals() and rb.plan:
    # normal run (Setup and Run plan ran in this session): run 09e if the plan says RUN, then its checks and outputs
    build_stage(STAGE_H)
    show_checks(STAGE_H)
    show_outputs(STAGE_H)
else:
    # No run plan in this session (e.g. after a kernel restart): show the latest 9e results read-only.
    # No AI calls are made and no run is recorded.
    import sys
    from pathlib import Path
    from IPython.display import display, Markdown
    if "rb" not in globals():
        from snowflake.snowpark.context import get_active_session
        WORKSPACE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sql" / "runbook_config.sql").exists())
        sys.path.insert(0, str(WORKSPACE / "lab_metadata"))
        import runbook_lib
        session = get_active_session()
        rb = runbook_lib.Runbook(session, WORKSPACE, globals().get("OPTIONS", {}))
        rb.apply_config()
    latest = session.sql("SELECT run_id FROM answer_judgments ORDER BY judged_at DESC LIMIT 1").collect()
    if not latest:
        display(Markdown("No 9e results yet: run the notebook from the top with `INCLUDE_ANSWER_EVAL=True`."))
    else:
        session.sql(f"SET runbook_run_id = '{latest[0][0]}'").collect()   # 9e output reads $runbook_run_id
        display(Markdown(f"No run plan in this session, so 09e was not run. Latest 9e results: run `{latest[0][0]}`"))
        for section, frame in rb.output_step("09e"):
            display(Markdown(f"**{section.id}** - {section.purpose} ({section.location})"))
            display(frame)

In [ ]:
# Tokens are measured when each step runs (AI_COUNT_TOKENS on the inputs, or the usage AI_COMPLETE returns),
# so they are available right away. Credits appear once a rate is set in ai_token_rates (credits per million
# tokens, from the Snowflake Service Consumption Table); until then the credits column says why it is empty.
# Cortex Search is billed for serving and indexing, not per query, so it shows a call count only.
if not halted():
    usage, elapsed = rb.run_cost()
    display(Markdown("**AI usage this run**"))
    display(usage if len(usage) else "(no AI step ran this run)")
    display(Markdown("**Elapsed time per step that ran**"))
    display(elapsed)

# Section 5: Record, compare, review and export
Record this run's per-question scores in `retrieval_question_scores_history`, compare them question by question (win/loss/tie with an exact sign test), sample the answers worth a human look, and read the reporting view `eval_run_report` that feeds the charts.

**Verdict rules.** A win means the variant ranked the first correct source higher (reciprocal rank @10; both missing is a tie). The overall verdict (`improved` / `regressed` / `inconclusive`, at p < 0.05) needs at least 15 shared questions on the `tune` set. A slice under 20 questions shows counts only. Every verdict is saved in `runbook_runs.summary`.

In [ ]:
if not halted():
    recorded = rb.record_scores()
    display(Markdown(f"Recorded per-question scores (eval fingerprint `{recorded['eval_fingerprint']}`, "
                     f"{recorded['answer_rows']} answer rows, {recorded['video_rows']} video rows)."))
    display(Markdown(f"Champion this run compares against: `{rb.champion_run_id or 'none yet'}`"))
    for result, slices, detail in rb.default_comparisons():
        display(Markdown(f"**{result['summary']}**  \nMRR@10 {result['mrr_a']} -> {result['mrr_b']} · "
                         f"recall@10 {result['recall_a']} -> {result['recall_b']}"))
        display(slices)
        display(detail[detail.outcome != "tie"])
    for v in rb.verdicts:
        if v.get("verdict") == "not comparable":
            display(Markdown(f"**{v['summary']}**"))

In [ ]:
# The champion is the run every later run is compared against (sign test, question by question).
# It changes only here, on purpose: promote a run after its verdict says it beat the champion.
# Every run records the champion it was compared against in runbook_runs.baseline_run_id.
# Set PROMOTE back to None after promoting: Run all would otherwise re-promote the same run every time.
PROMOTE = None   # e.g. ("<run_id>", "VECTOR improved vs champion: 9 wins, 1 loss, p = 0.02")
# Promoted 2026-11-25: ("9dfccaf7f3c4", "desc v2 image-aware: MRR 0.782→0.900, 4W/0L")
if PROMOTE:
    display(Markdown(rb.promote(*PROMOTE)))
display(session.sql("""SELECT run_id, experiment_name, promoted_at, promoted_reason
                       FROM runbook_runs WHERE promoted_at IS NOT NULL ORDER BY promoted_at DESC""").to_pandas())

## Human review: grade the judge's answers
1. Run **Human review: sample answers to grade**. The first time for a run, it puts the sample into `hitl_reviews` as **pending** rows (no score yet); running it again adds nothing. It shows each row's `QUESTION`, `ANSWER`, `SOURCES` and the judge's scores, pending rows first.
2. In **Human review: save my grades**, add one line per answer you graded to `MY_GRADES`: your 1-5 score for how well the answer is supported by its sources, whether refusing (or not) was right, and a note. Grade a few at a time if you like.
3. Run that cell. Each line updates its row in `hitl_reviews` (saving again overwrites, never duplicates), then it shows what is still pending and how far the judge is from you (`mean_abs_diff`, `within_1`). Large gaps mean the judge prompt needs work before its scores can be trusted.

Scale: 5 = every claim supported and cited, 3 = partly supported, 1 = unsupported, or a wrong refusal.

What is left to grade, any time: `SELECT * FROM hitl_reviews WHERE human_score IS NULL;`

In [ ]:
# Rows where your eyes add the most: the judge disagrees with retrieval (right source at #1, low groundedness),
# a wrong refusal, near misses (right source at #2-#5), and a fixed calibration sample per run.
# The first run for a review run queues them in hitl_reviews as pending rows; rerunning adds nothing.
# It also works after a kernel restart: if Setup has not run in this session, it loads the engine
# read-only (applies sql/runbook_config.sql, records no new run).
import sys
from pathlib import Path
from IPython.display import display, Markdown
if "rb" not in globals():
    from snowflake.snowpark.context import get_active_session
    WORKSPACE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sql" / "runbook_config.sql").exists())
    sys.path.insert(0, str(WORKSPACE / "lab_metadata"))
    import runbook_lib
    session = get_active_session()
    rb = runbook_lib.Runbook(session, WORKSPACE, globals().get("OPTIONS", {}))
    rb.apply_config()

# Review the latest run that has judged answers (step 09e); set a run_id here to review another run.
latest = session.sql("SELECT run_id FROM answer_judgments ORDER BY judged_at DESC LIMIT 1").collect()
REVIEW_RUN_ID = latest[0][0] if latest else None

if rb.halted_reason:
    display(Markdown(f"Skipped: {rb.halted_reason}"))
elif REVIEW_RUN_ID is None:
    display(Markdown("No answers to review yet: run with `INCLUDE_ANSWER_EVAL=True` (step 09e)."))
else:
    queue = rb.hitl_sample(n_random=5, run_id=REVIEW_RUN_ID)
    queue.columns = [c.lower() for c in queue.columns]
    pending = queue[queue.status == "pending"]
    display(Markdown(f"Run `{REVIEW_RUN_ID}`: {len(pending)} pending, {len(queue) - len(pending)} graded"))
    display(queue)
    if len(pending):
        # Starter lines for the next cell: copy the ones you grade into MY_GRADES.
        print("\n".join(f'    "{r.question_id}": (None, None, ""),   # {r.sample_reason}; judge groundedness {r.groundedness}'
                        for r in pending.itertuples()))

In [ ]:
# One line per answer you graded:  "Q08": (score 1-5, refusal was right True/False, "note"),
# Saving the same question again overwrites its grade. Lines whose score is None are skipped.
# Grades are saved for REVIEW_RUN_ID, the run shown by the cell above.
MY_GRADES = {
    # "Q21": (1, False, "refused, but slide 23 of the day 2 lecture answers it"),
    # "Q08": (5, True, "cites the FAQ and the kickoff slide correctly"),
}
graded = {qid: g for qid, g in MY_GRADES.items() if g[0] is not None}
for qid, (score, refusal_ok, note) in graded.items():
    rb.save_review(qid, human_score=score, human_refusal_ok=refusal_ok, notes=note or None, run_id=REVIEW_RUN_ID)
queue = rb.hitl_queue(REVIEW_RUN_ID)
queue.columns = [c.lower() for c in queue.columns]
display(Markdown(f"Saved {len(graded)} grades for run `{REVIEW_RUN_ID}`; "
                 f"{int((queue.status == 'pending').sum())} still pending."))
display(queue[["question_id", "status", "sample_reason", "groundedness", "human_score", "human_refusal_ok", "notes"]])
display(Markdown("**Judge vs your grades, all reviewed answers so far**"))
display(rb.judge_agreement())

In [ ]:
%%sql -r run_report
-- One row per run x method x slice. verdict_allowed is FALSE for slices too small to pick a winner.
-- Charts read this view: progression by experiment_name, accuracy vs ai_credits / elapsed_s, and slice counts.
SELECT run_id, experiment_name, method, eval_set, slice_kind, slice, questions, verdict_allowed,
       hit_at_1, hit_at_5, mrr_at_10, recall_at_10, ndcg_at_10,
       groundedness, refusal_accuracy, ai_input_tokens, ai_credits, elapsed_s,
       eval_fingerprint, text_embed_model, source_freshness, verdicts
FROM eval_run_report
WHERE run_id = '{{RUN_ID}}'
ORDER BY slice_kind, slice, method;

## Record the run and clean up
Saves this run's scores to `retrieval_scores_history`, closes the run in `runbook_runs`, and lists every step decision for this run. The baseline search service keeps refreshing daily on the warehouse until you drop it: set `CONFIRM_DROP=True` and rerun this cell when you're done comparing.

In [ ]:
if not halted():
    # scores were recorded in 'Record scores and compare (sign test)'; the verdicts go into runbook_runs.summary
    summary = rb.run_summary()
    rb.end_run("SUCCESS", {"steps_run": int((summary.STATUS == "SUCCESS").sum()),
                            "steps_skipped": int(summary.STATUS.str.startswith("SKIPPED").sum())})
    display(Markdown(f"**Run `{rb.run_id}` recorded.**"))
    display(summary)
    if OPTIONS["MODE"] in ("score_existing", "full_experiment"):
        display(Markdown(rb.drop_baseline_service()))

### Look up past runs
```sql
SELECT * FROM runbook_runs ORDER BY started_at DESC;
SELECT step, decision, status, reasons, checks, elapsed_ms FROM runbook_step_runs WHERE run_id = '<run_id>' ORDER BY started_at;
-- question-grain history (one row per run, method, question) and the method-grain view over it
SELECT * FROM retrieval_question_scores_history WHERE run_id = '<run_id>' ORDER BY method, question_id;
SELECT * FROM retrieval_scores_history ORDER BY recorded_at DESC, method;
SELECT * FROM retrieval_scores_history_legacy ORDER BY recorded_at DESC;   -- the 15 rows from before the question-grain fact (frozen)
-- reporting layer for charts: run x method x slice, with counts, fingerprint, config, cost, freshness, verdicts
SELECT * FROM eval_run_report WHERE slice_kind = 'all' ORDER BY started_at DESC, method;
SELECT * FROM runbook_ai_usage WHERE run_id = '<run_id>';
SELECT * FROM hitl_reviews ORDER BY reviewed_at DESC;
SELECT * FROM runbook_source_freshness ORDER BY checked_at DESC;
SELECT run_id, source, COUNT(*) FROM slide_descriptions_history GROUP BY 1, 2;
```